# Bag of Words Exploration Notebook

## 1. Setup 

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from stop_words import stop_word_list

# Directory containing .txt files
TEXT_DIR = "text" 

# Directory containing other metadata
DATA_DIR = "data"

## 2. Load documents 

In [ ]:
filepaths = glob.glob(os.path.join(TEXT_DIR, "*.txt"))

corpus = []
filenames = []

for fp in filepaths:
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        corpus.append(f.read())
        filenames.append(os.path.basename(fp))

print(f"Loaded {len(corpus)} documents")

In [ ]:
#nltk.download('punkt_tab')

stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # keep only letters
    tokens = word_tokenize(text)
    stems = [stemmer.stem(t) for t in tokens]
    return " ".join(stems)

corpus_stemmed = [preprocess(doc) for doc in corpus]

## 3. Create Bag-of-Words representation 

In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

vectorizer = CountVectorizer(
    strip_accents='unicode',
    stop_words=stop_word_list,
    max_df=1.0,
    min_df=5
)

X = vectorizer.fit_transform(corpus_stemmed)

print("Matrix shape:", X.shape)

## 4. Convert to DataFrame for exploration 

In [ ]:
feature_names = vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(X.toarray(), columns=feature_names, index=filenames)

bow_df.head()

## 5. Most common words overall 

In [ ]:
word_counts = bow_df.sum(axis=0).sort_values(ascending=False)

print("Top 20 words:")
print(word_counts.head(40))

## 6. Most important words per document 

In [ ]:
def top_words_per_doc(df, n=10):
    results = {}
    for doc in df.index:
        top_words = df.loc[doc].sort_values(ascending=False).head(n)
        results[doc] = top_words[top_words > 0]
    return results

per_doc = top_words_per_doc(bow_df)

# Example: show one document
example_doc = list(per_doc.keys())[0]
print(f"\nTop words in {example_doc}:")
print(per_doc[example_doc])

## 7. Document similarity (cosine similarity) 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X)

sim_df = pd.DataFrame(similarity_matrix, index=filenames, columns=filenames)

sim_df.head()

## 8. Find most similar documents 

In [ ]:
def most_similar(df, doc_name, top_n=5):
    sims = df.loc[doc_name].drop(doc_name)
    return sims.sort_values(ascending=False).head(top_n)

print(f"\nDocuments most similar to {example_doc}:")
print(most_similar(sim_df, example_doc))

## 9. Optional: visualize top words 

In [ ]:
import matplotlib.pyplot as plt

word_counts.head(20).plot(kind='bar')
plt.title("Top 20 Words Across Corpus")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Save outputs 

In [ ]:
bow_df.to_csv("outputs/bow_matrix.csv")
sim_df.to_csv("outputs/document_similarity.csv")

print("Saved outputs: bow_matrix.csv, document_similarity.csv")

# Sanity Checks

### 1. Locate document pairs with highest similarity.

In [ ]:
# Grab just the upper triangle (excluding diagonal).
# This eliminates documents trivial pairs (like documents being similar to themselves)
upper_tri = np.triu(np.ones(sim_df.shape), k=1).astype(bool)

filtered = sim_df.where(upper_tri)

In [ ]:
# We'll pull the most similar 5 pairs

top_pairs = (
    filtered.stack()
    .sort_values(ascending=False)
    .head(5)
)
top_pairs

In [ ]:
duplicates_99 = []

threshold = 0.99  # adjust (0.9–0.99 depending on strictness)

n = similarity_matrix.shape[0]

for i in range(n):
    for j in range(i+1, n):
        if similarity_matrix[i, j] > threshold:
            duplicates_99.append((filenames[i], filenames[j], similarity_matrix[i, j]))

print("Near-duplicate pairs:")
for d in duplicates_99:
    print(d)

In [ ]:
duplicates_75 = []

threshold = 0.75  # adjust (0.9–0.99 depending on strictness)

n = similarity_matrix.shape[0]

for i in range(n):
    for j in range(i+1, n):
        if 0.99 > similarity_matrix[i, j] > threshold:
            duplicates_75.append((filenames[i], filenames[j], similarity_matrix[i, j]))

print("Near-duplicate pairs:")
for d in duplicates_75:
    print(d)